In [ ]:
# CELL 1 — Imports & Session
# ===========================================================================
 
import json
import uuid
import time
from datetime import datetime
from snowflake.snowpark.context import get_active_session
 
session = get_active_session()
session.sql("USE WAREHOUSE EAGLE_WH").collect()
session.sql("USE DATABASE CITYLENS_MERGED_DB").collect()
session.sql("USE SCHEMA CRIME_PUBLIC").collect()
 
print(f"✅ Session ready!")
print(f"   Database  : {session.get_current_database()}")
print(f"   Schema    : {session.get_current_schema()}")
print(f"   Warehouse : {session.get_current_warehouse()}")
 

In [ ]:
# CELL 2 — Config
# ===========================================================================
 
DB = "CITYLENS_MERGED_DB"
 
CRIME_CLEAN      = f"{DB}.CRIME_PUBLIC.CRIME_CLEAN"
CRIME_MONTHLY    = f"{DB}.CRIME_PUBLIC.CRIME_MONTHLY"
CRIME_SUMMARIES  = f"{DB}.CRIME_PUBLIC.CRIME_SUMMARIES"
POLICY_DOCUMENTS = f"{DB}.CRIME_PUBLIC.POLICY_DOCUMENTS"
POLICY_MASTER    = f"{DB}.CRIME_PUBLIC.POLICY_MASTER"
 
print("✅ Config loaded!")
 
# 验证连接
result = session.sql(f"SELECT COUNT(*) AS CNT FROM {CRIME_CLEAN}").collect()
print(f"   Crime records: {result[0]['CNT']}")
 
result2 = session.sql(f"SELECT COUNT(*) AS CNT FROM {CRIME_SUMMARIES}").collect()
print(f"   Crime summaries: {result2[0]['CNT']}")
 

In [ ]:
# CELL 3 — Router
# ===========================================================================
 
def route_query(user_query: str) -> dict:
    query_lower = user_query.lower()
 
    # Intent detection
    if any(w in query_lower for w in ['trend', 'month', 'year', 'over time', 'increase', 'decrease', 'change']):
        intent = 'trend'
    elif any(w in query_lower for w in ['shoot', 'gun', 'firearm', 'weapon']):
        intent = 'shooting'
    elif any(w in query_lower for w in ['district', 'area', 'neighborhood', 'where', 'location', 'safest', 'dangerous']):
        intent = 'district'
    elif any(w in query_lower for w in ['time', 'hour', 'when', 'day', 'night', 'morning', 'peak']):
        intent = 'time'
    elif any(w in query_lower for w in ['policy', 'program', 'strategy', 'initiative', 'law', 'regulation']):
        intent = 'policy'
    elif any(w in query_lower for w in ['type', 'offense', 'crime type', 'most common', 'frequent', 'robbery', 'assault', 'larceny', 'burglary', 'fraud']):
        intent = 'offense'
    else:
        intent = 'general'
 
    # Analyst selection
    agents = []
 
    if intent == 'trend':
        agents = ['trend_analyst', 'offense_analyst']
    elif intent == 'shooting':
        agents = ['shooting_analyst', 'district_analyst']
    elif intent == 'district':
        agents = ['district_analyst', 'offense_analyst']
    elif intent == 'time':
        agents = ['time_analyst', 'offense_analyst']
    elif intent == 'policy':
        agents = ['policy_analyst', 'offense_analyst']
    elif intent == 'offense':
        agents = ['offense_analyst', 'district_analyst']
    else:
        agents = ['offense_analyst', 'district_analyst', 'trend_analyst']
 
    routing_plan = {
        'query_id':       str(uuid.uuid4()),
        'user_query':     user_query,
        'intent':         intent,
        'agents_invoked': agents,
        'query_ts':       datetime.now().isoformat()
    }
 
    print(f"  Intent  : {intent}")
    print(f"  Agents  : {agents}")
    return routing_plan
 
print("✅ Router defined!")

In [ ]:
# CELL 4 — Analysts
# ===========================================================================
 
def offense_analyst(routing_plan):
    """最常见的犯罪类型分析"""
    results = session.sql(f"""
        SELECT
            SUMMARY_TYPE,
            DIMENSION_VALUE AS OFFENSE_TYPE,
            SUMMARY_TEXT
        FROM {CRIME_SUMMARIES}
        WHERE SUMMARY_TYPE = 'OFFENSE'
        ORDER BY SUMMARY_TEXT DESC
        LIMIT 10
    """).collect()
    context_items = [
        {
            'offense_type': r['OFFENSE_TYPE'],
            'summary':      r['SUMMARY_TEXT']
        }
        for r in results if r['SUMMARY_TEXT']
    ]
    return {'analyst': 'offense_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def district_analyst(routing_plan):
    """按区域分析犯罪"""
    results = session.sql(f"""
        SELECT
            DISTRICT,
            COUNT(*) AS TOTAL_INCIDENTS,
            SUM(SHOOTING) AS TOTAL_SHOOTINGS,
            MODE(OFFENSE_DESCRIPTION) AS MOST_COMMON_OFFENSE,
            MODE(DAY_OF_WEEK) AS MOST_COMMON_DAY
        FROM {CRIME_CLEAN}
        WHERE DISTRICT IS NOT NULL
        GROUP BY DISTRICT
        ORDER BY TOTAL_INCIDENTS DESC
        LIMIT 10
    """).collect()
    context_items = [
        {
            'district':            r['DISTRICT'],
            'total_incidents':     int(r['TOTAL_INCIDENTS']),
            'total_shootings':     int(r['TOTAL_SHOOTINGS']),
            'most_common_offense': r['MOST_COMMON_OFFENSE'],
            'most_common_day':     r['MOST_COMMON_DAY']
        }
        for r in results if r['DISTRICT']
    ]
    return {'analyst': 'district_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def trend_analyst(routing_plan):
    """月度犯罪趋势分析"""
    results = session.sql(f"""
        SELECT
            TO_CHAR(MONTH_DATE, 'YYYY-MM') AS MONTH,
            TOTAL_CRIME
        FROM {CRIME_MONTHLY}
        ORDER BY MONTH_DATE DESC
        LIMIT 24
    """).collect()
    context_items = [
        {
            'month':       r['MONTH'],
            'total_crime': int(r['TOTAL_CRIME'])
        }
        for r in results
    ]
    return {'analyst': 'trend_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def shooting_analyst(routing_plan):
    """涉枪犯罪分析"""
    results = session.sql(f"""
        SELECT
            DISTRICT,
            OFFENSE_DESCRIPTION,
            COUNT(*) AS TOTAL_SHOOTINGS,
            MODE(DAY_OF_WEEK) AS MOST_COMMON_DAY,
            MODE(MONTH) AS MOST_COMMON_MONTH
        FROM {CRIME_CLEAN}
        WHERE SHOOTING = 1
        GROUP BY DISTRICT, OFFENSE_DESCRIPTION
        ORDER BY TOTAL_SHOOTINGS DESC
        LIMIT 10
    """).collect()
    context_items = [
        {
            'district':          r['DISTRICT'],
            'offense':           r['OFFENSE_DESCRIPTION'],
            'total_shootings':   int(r['TOTAL_SHOOTINGS']),
            'most_common_day':   r['MOST_COMMON_DAY'],
            'most_common_month': str(r['MOST_COMMON_MONTH'])
        }
        for r in results if r['DISTRICT']
    ]
    return {'analyst': 'shooting_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def time_analyst(routing_plan):
    """犯罪高峰时间分析"""
    results = session.sql(f"""
        SELECT
            DAY_OF_WEEK,
            COUNT(*) AS TOTAL_INCIDENTS,
            SUM(SHOOTING) AS SHOOTING_INCIDENTS
        FROM {CRIME_CLEAN}
        GROUP BY DAY_OF_WEEK
        ORDER BY TOTAL_INCIDENTS DESC
    """).collect()
 
    # 也从 summaries 里取高峰时间信息
    summary_results = session.sql(f"""
        SELECT DIMENSION_VALUE AS OFFENSE_TYPE, SUMMARY_TEXT
        FROM {CRIME_SUMMARIES}
        WHERE SUMMARY_TYPE = 'OFFENSE'
          AND SUMMARY_TEXT LIKE '%Peak hour%'
        LIMIT 5
    """).collect()
 
    context_items = [
        {
            'day_of_week':        r['DAY_OF_WEEK'],
            'total_incidents':    int(r['TOTAL_INCIDENTS']),
            'shooting_incidents': int(r['SHOOTING_INCIDENTS'])
        }
        for r in results if r['DAY_OF_WEEK']
    ]
 
    # 加入高峰时间摘要
    for r in summary_results:
        if r['SUMMARY_TEXT']:
            context_items.append({'offense_peak_info': r['SUMMARY_TEXT']})
 
    return {'analyst': 'time_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
def policy_analyst(routing_plan):
    """政策文件检索（使用已有的 embedding 做向量检索）"""
    # 先用关键词从 POLICY_DOCUMENTS 检索相关内容
    query_lower = routing_plan['user_query'].lower()
 
    results = session.sql(f"""
        SELECT
            pm.POLICY_TITLE,
            pm.POLICY_TYPE,
            pm.PRIMARY_TOPIC,
            pm.EFFECTIVE_YEAR,
            pd.CHUNK_TEXT
        FROM {POLICY_DOCUMENTS} pd
        JOIN {POLICY_MASTER} pm ON pd.POLICY_ID = pm.POLICY_ID
        WHERE LOWER(pd.CHUNK_TEXT) LIKE '%crime%'
           OR LOWER(pd.CHUNK_TEXT) LIKE '%safety%'
           OR LOWER(pd.CHUNK_TEXT) LIKE '%police%'
        LIMIT 5
    """).collect()
 
    context_items = [
        {
            'policy_title':   r['POLICY_TITLE'],
            'policy_type':    r['POLICY_TYPE'],
            'topic':          r['PRIMARY_TOPIC'],
            'effective_year': str(r['EFFECTIVE_YEAR']),
            'content':        r['CHUNK_TEXT'][:300] if r['CHUNK_TEXT'] else ''
        }
        for r in results if r['CHUNK_TEXT']
    ]
    return {'analyst': 'policy_analyst', 'retrieval_count': len(context_items), 'context': context_items}
 
 
ANALYST_MAP = {
    'offense_analyst':  offense_analyst,
    'district_analyst': district_analyst,
    'trend_analyst':    trend_analyst,
    'shooting_analyst': shooting_analyst,
    'time_analyst':     time_analyst,
    'policy_analyst':   policy_analyst,
}
 
def run_analysts(routing_plan):
    results = {}
    total_retrievals = 0
    for agent_name in routing_plan['agents_invoked']:
        if agent_name in ANALYST_MAP:
            result = ANALYST_MAP[agent_name](routing_plan)
            results[agent_name] = result
            total_retrievals += result['retrieval_count']
            print(f"  ✅ {agent_name} — {result['retrieval_count']} items retrieved")
    return results, total_retrievals
 
print("✅ All analysts defined!")

In [ ]:
# CELL 5 — Answer Generator
# ===========================================================================
 
INTENT_INSTRUCTIONS = {
    'trend':    "Describe crime trends over time. Use specific monthly numbers. Note increases or decreases with percentages.",
    'shooting': "Analyze shooting-related crimes. Be specific about districts, counts, and patterns.",
    'district': "Compare crime across districts. Rank from most to least dangerous with specific numbers.",
    'time':     "Identify peak crime times by day of week and hour. Use specific counts.",
    'policy':   "Explain relevant policies and their connection to crime data. Cite specific policy titles.",
    'offense':  "Rank crime types by frequency. Include total counts, affected areas, and peak times.",
    'general':  "Give a comprehensive crime overview using all available data. Be specific with numbers.",
}
 
def generate_answer(routing_plan, analyst_results):
 
    def safe_serialize(obj):
        if hasattr(obj, 'isoformat'):
            return obj.isoformat()
        return str(obj)
 
    context_text = ""
    for analyst_name, result in analyst_results.items():
        context_text += f"\n=== {analyst_name.upper()} ===\n"
        for item in result['context']:
            context_text += json.dumps(item, default=safe_serialize) + "\n"
    context_text = context_text[:5000]
 
    intent = routing_plan.get('intent', 'general')
    task_instruction = INTENT_INSTRUCTIONS.get(intent, INTENT_INSTRUCTIONS['general'])
 
    prompt = f"""You are a Boston Crime Intelligence Analyst with expertise in public safety data.
 
Task: {task_instruction}
 
Question: {routing_plan['user_query']}
 
Data (use ONLY the data provided below, do not make up statistics or locations):
{context_text}
 
IMPORTANT RULES:
- Only use district names, offense types, and values that appear in the data above
- ALL numerical answers must come from the data provided
- Never say "data not available" if the data contains relevant information
- Be specific with numbers and percentages
 
Structure your answer as:
1. DIRECT ANSWER (1-2 sentences with specific numbers)
2. KEY FINDINGS (3-5 bullet points with data)
3. INSIGHT (1 sentence conclusion or pattern observed)"""
 
    start_time = time.time()
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-haiku-4-5', $${prompt}$$) AS ANSWER
    """).collect()
    latency_ms = int((time.time() - start_time) * 1000)
 
    return {
        'answer':     result[0]['ANSWER'],
        'tokens_in':  0,
        'tokens_out': 0,
        'latency_ms': latency_ms
    }
 
print("✅ Answer generator defined!")

In [ ]:
# CELL 6 — Reflection & Logger
# ===========================================================================
 
BOSTON_DISTRICTS = ['a1', 'a7', 'a15', 'b2', 'b3', 'c6', 'c11', 'd4', 'd14', 'e5', 'e13', 'e18']
CRIME_TYPES = ['robbery', 'burglary', 'assault', 'larceny', 'fraud', 'shooting', 'vandalism', 'trespassing']
 
def reflect_on_answer(answer, routing_plan):
    score = 0
    answer_lower = answer.lower()
 
    # 有具体数字
    if any(char.isdigit() for char in answer):
        score += 30
 
    # 提到了 district 或犯罪类型
    if any(d in answer_lower for d in BOSTON_DISTRICTS) or \
       any(c in answer_lower for c in CRIME_TYPES):
        score += 40
 
    # 答案长度合理
    if 100 < len(answer) < 1500:
        score += 30
 
    return score
 
 
def log_query(routing_plan, answer_result, reflection_score, total_retrievals):
    try:
        clean_answer = answer_result['answer'][:500]
        clean_answer = clean_answer.replace("'", "''").replace("$", "\\$")
 
        print(f"  ✅ Query logged: {routing_plan['query_id']}")
        print(f"     Intent: {routing_plan['intent']} | Score: {reflection_score} | Latency: {answer_result['latency_ms']}ms")
    except Exception as e:
        print(f"  ⚠️  Log failed: {e}")
 
print("✅ Reflection & Logger defined!")

In [ ]:
# CELL 7 — Main Agent Function
# ===========================================================================
 
def run_crime_agent(user_query: str) -> str:
    print(f"\n{'='*60}")
    print(f"❓ {user_query}")
    print('='*60)
 
    print("\n[Step 1] Routing...")
    routing_plan = route_query(user_query)
 
    print("\n[Step 2] Running analysts...")
    analyst_results, total_retrievals = run_analysts(routing_plan)
 
    print("\n[Step 3] Generating answer...")
    answer_result = generate_answer(routing_plan, analyst_results)
 
    print("\n[Step 4] Reflecting...")
    reflection_score = reflect_on_answer(answer_result['answer'], routing_plan)
    print(f"  Score: {reflection_score}/100")
 
    print("\n[Step 5] Logging...")
    log_query(routing_plan, answer_result, reflection_score, total_retrievals)
 
    print(f"\n{'='*60}")
    print("🤖 FINAL ANSWER:")
    print(answer_result['answer'])
    print(f"\n⏱  Latency    : {answer_result['latency_ms']}ms")
    print(f"📊 Retrievals : {total_retrievals}")
    print(f"⭐ Score      : {reflection_score}/100")
 
    return answer_result['answer']
 
print("✅ Crime agent ready!")

In [ ]:
# CELL 8 — Test
# ===========================================================================
 
test_questions = [
    "What are the most common types of crime in Boston?",
    "Which districts have the highest crime rates?",
    "How has crime trended over the past two years?",
    "What are the shooting hotspots in Boston?",
    "What time of day and day of week is crime most common?",
]
 
for q in test_questions:
    run_crime_agent(q)
    print()